In [13]:


import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
import string

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    AutoModel
)

from sklearn.metrics import classification_report

# =========================================================
# 1. CHECK DEVICE
# =========================================================
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# =========================================================
# 2. LOAD DATA (XML + CSV)
# =========================================================
def xml_to_rows(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    rows = []
    for sentence in root.findall("sentence"):
        sid = sentence.get("id")
        text = sentence.find("text").text

        aspect_terms = sentence.find("aspectTerms")
        if aspect_terms is not None:
            for aspect in aspect_terms.findall("aspectTerm"):
                rows.append({
                    "id": sid,
                    "sentence": text,
                    "aspect": aspect.get("term"),
                    "polarity": aspect.get("polarity")
                })
    return pd.DataFrame(rows)


xml_df = xml_to_rows("./Restaurants_Train_v2.xml")

csv_df = pd.read_csv("./Laptop_Train_v2.csv", encoding="utf-8")
csv_df = csv_df[["id", "Sentence", "Aspect Term", "polarity"]]
csv_df.columns = ["id", "sentence", "aspect", "polarity"]

final_df = pd.concat([csv_df, xml_df], ignore_index=True)

final_df["polarity"] = final_df["polarity"].str.strip().str.upper()
final_df["aspect"] = final_df["aspect"].str.replace('"', '')

final_df = final_df.head(6000)

# =========================================================
# 3. TOKENIZATION + BIOES TAGGING
# =========================================================
def tokenize(text):
    return [
        t.strip(string.punctuation).lower()
        for t in text.split()
        if t.strip(string.punctuation)
    ]

def find_span(tokens, aspect_tokens):
    n, m = len(tokens), len(aspect_tokens)
    for i in range(n - m + 1):
        if tokens[i:i+m] == aspect_tokens:
            return i, i + m - 1
    return -1, -1

def build_tags(tokens, start, end, polarity):
    tags = ["O"] * len(tokens)

    if start == -1:
        return tags

    length = end - start + 1

    if length == 1:
        tags[start] = f"S-{polarity}"
    else:
        tags[start] = f"B-{polarity}"
        for i in range(start + 1, end):
            tags[i] = f"I-{polarity}"
        tags[end] = f"E-{polarity}"

    return tags

def convert(df):
    dataset = []

    for _, row in df.iterrows():
        sentence = row["sentence"]
        aspect = row["aspect"]
        polarity = row["polarity"]

        tokens = tokenize(sentence)
        aspect_tokens = tokenize(aspect)

        start, end = find_span(tokens, aspect_tokens)
        tags = build_tags(tokens, start, end, polarity)

        dataset.append(list(zip(tokens, tags)))

    return dataset

bioes_data = convert(final_df)

# =========================================================
# 4. LABELS
# =========================================================
labels = [
    "O",
    "B-POSITIVE", "I-POSITIVE", "E-POSITIVE", "S-POSITIVE",
    "B-NEGATIVE", "I-NEGATIVE", "E-NEGATIVE", "S-NEGATIVE",
    "B-NEUTRAL", "I-NEUTRAL", "E-NEUTRAL", "S-NEUTRAL",
    "B-CONFLICT", "I-CONFLICT", "E-CONFLICT", "S-CONFLICT"
]

label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}

# =========================================================
# 5. DATASET CREATION
# =========================================================
def prepare_dataset(bioes_data):
    tokens_list = []
    labels_list = []

    for sent in bioes_data:
        tokens_list.append([t for t,l in sent])
        labels_list.append([l for t,l in sent])

    return Dataset.from_dict({
        "tokens": tokens_list,
        "labels": labels_list
    })

dataset = prepare_dataset(bioes_data)
dataset = dataset.train_test_split(test_size=0.2)

# =========================================================
# 6. TOKENIZER
# =========================================================
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_and_align(examples):
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True
    )

    labels_batch = []

    for i, word_labels in enumerate(examples["labels"]):
        word_ids = tokenized.word_ids(batch_index=i)

        previous = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)

            elif word_idx != previous:
                label_ids.append(label2id[word_labels[word_idx]])

            else:
                label_ids.append(label2id[word_labels[word_idx]])

            previous = word_idx

        labels_batch.append(label_ids)

    tokenized["labels"] = labels_batch
    return tokenized

tokenized_dataset = dataset.map(tokenize_and_align, batched=True)

# =========================================================
# 7. BERT + BiLSTM MODEL
# =========================================================
class BERT_BiLSTM(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()

        self.bert = AutoModel.from_pretrained(model_name)

        self.lstm = nn.LSTM(
            input_size=self.bert.config.hidden_size,
            hidden_size=256,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(256 * 2, num_labels)

    def forward(self, input_ids=None, attention_mask=None, labels=None):

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        x = outputs.last_hidden_state

        x, _ = self.lstm(x)
        x = self.dropout(x)

        logits = self.classifier(x)

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fn(
                logits.view(-1, logits.shape[-1]),
                labels.view(-1)
            )

        return {"loss": loss, "logits": logits}

model = BERT_BiLSTM(model_name, len(labels))

# =========================================================
# 8. TRAINING SETUP
# =========================================================
data_collator = DataCollatorForTokenClassification(tokenizer)

training_args = TrainingArguments(
    output_dir="./absa_bilstm_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    report_to="none"
)

# =========================================================
# 9. METRICS - FIXED VERSION
# =========================================================
def compute_metrics(p):
    preds, labels = p
    preds = np.argmax(preds, axis=2)

    # Flatten all predictions and labels
    flat_true_labels = []
    flat_true_preds = []

    for i in range(len(preds)):
        for j in range(len(preds[i])):
            if labels[i][j] != -100:  # Skip padding tokens
                flat_true_labels.append(id2label[labels[i][j]])
                flat_true_preds.append(id2label[preds[i][j]])

    return {
        "report": classification_report(flat_true_labels, flat_true_preds, digits=4)
    }

# =========================================================
# 10. TRAINER
# =========================================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

# =========================================================
# 11. SAVE MODEL
# =========================================================
trainer.save_model("./absa_bilstm_model")
tokenizer.save_pretrained("./absa_bilstm_model")

print("Training completed and model saved successfully!")

CUDA available: True
GPU: NVIDIA GeForce RTX 4080


Map:   0%|          | 0/4800 [00:00<?, ? examples/s]

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Report
1,No log,0.354379,precision recall f1-score support B-CONFLICT 0.0000 0.0000 0.0000 4 B-NEGATIVE 0.0000 0.0000 0.0000 121 B-NEUTRAL 0.0000 0.0000 0.0000 93 B-POSITIVE 0.3333 0.0045 0.0088 224 E-CONFLICT 0.0000 0.0000 0.0000 4 E-NEGATIVE 0.0000 0.0000 0.0000 107 E-NEUTRAL 0.0000 0.0000 0.0000 83 E-POSITIVE 0.0000 0.0000 0.0000 204 I-NEGATIVE 0.0000 0.0000 0.0000 30 I-NEUTRAL 0.0000 0.0000 0.0000 50 I-POSITIVE 0.0000 0.0000 0.0000 104 O 0.9203 0.9985 0.9578 22158 S-CONFLICT 0.0000 0.0000 0.0000 21 S-NEGATIVE 0.0000 0.0000 0.0000 297 S-NEUTRAL 0.0000 0.0000 0.0000 203 S-POSITIVE 0.3649 0.1392 0.2016 553 accuracy 0.9153 24256 macro avg 0.1012 0.0714 0.0730 24256 weighted avg 0.8521 0.9153 0.8796 24256
2,0.443973,0.313679,precision recall f1-score support B-CONFLICT 0.0000 0.0000 0.0000 4 B-NEGATIVE 0.0000 0.0000 0.0000 121 B-NEUTRAL 0.0000 0.0000 0.0000 93 B-POSITIVE 0.4000 0.0089 0.0175 224 E-CONFLICT 0.0000 0.0000 0.0000 4 E-NEGATIVE 0.0000 0.0000 0.0000 107 E-NEUTRAL 0.0000 0.0000 0.0000 83 E-POSITIVE 0.4286 0.0441 0.0800 204 I-NEGATIVE 0.0000 0.0000 0.0000 30 I-NEUTRAL 0.0000 0.0000 0.0000 50 I-POSITIVE 0.0000 0.0000 0.0000 104 O 0.9250 0.9957 0.9590 22158 S-CONFLICT 0.0000 0.0000 0.0000 21 S-NEGATIVE 0.5068 0.1246 0.2000 297 S-NEUTRAL 0.0000 0.0000 0.0000 203 S-POSITIVE 0.4754 0.2622 0.3380 553 accuracy 0.9175 24256 macro avg 0.1710 0.0897 0.0997 24256 weighted avg 0.8693 0.9175 0.8871 24256
3,0.443973,0.301511,precision recall f1-score support B-CONFLICT 0.0000 0.0000 0.0000 4 B-NEGATIVE 0.5263 0.1653 0.2516 121 B-NEUTRAL 0.0000 0.0000 0.0000 93 B-POSITIVE 0.5400 0.1205 0.1971 224 E-CONFLICT 0.0000 0.0000 0.0000 4 E-NEGATIVE 0.5676 0.1963 0.2917 107 E-NEUTRAL 0.0000 0.0000 0.0000 83 E-POSITIVE 0.5250 0.2059 0.2958 204 I-NEGATIVE 0.0000 0.0000 0.0000 30 I-NEUTRAL 0.0000 0.0000 0.0000 50 I-POSITIVE 0.0000 0.0000 0.0000 104 O 0.9355 0.9905 0.9622 22158 S-CONFLICT 0.0000 0.0000 0.0000 21 S-NEGATIVE 0.4646 0.3098 0.3717 297 S-NEUTRAL 0.4286 0.0148 0.0286 203 S-POSITIVE 0.4974 0.3472 0.4089 553 accuracy 0.9212 24256 macro avg 0.2803 0.1469 0.1755 24256 weighted avg 0.8897 0.9212 0.8999 24256
4,0.281973,0.306166,precision recall f1-score support B-CONFLICT 0.0000 0.0000 0.0000 4 B-NEGATIVE 0.4262 0.2149 0.2857 121 B-NEUTRAL 0.0000 0.0000 0.0000 93 B-POSITIVE 0.5517 0.1429 0.2270 224 E-CONFLICT 0.0000 0.0000 0.0000 4 E-NEGATIVE 0.5102 0.2336 0.3205 107 E-NEUTRAL 0.0000 0.0000 0.0000 83 E-POSITIVE 0.5610 0.2255 0.3217 204 I-NEGATIVE 0.0000 0.0000 0.0000 30 I-NEUTRAL 0.0000 0.0000 0.0000 50 I-POSITIVE 0.0000 0.0000 0.0000 104 O 0.9380 0.9852 0.9610 22158 S-CONFLICT 0.0000 0.0000 0.0000 21 S-NEGATIVE 0.4286 0.3737 0.3993 297 S-NEUTRAL 0.2000 0.0049 0.0096 203 S-POSITIVE 0.4658 0.3942 0.4270 553 accuracy 0.9189 24256 macro avg 0.2551 0.1609 0.1845 24256 weighted avg 0.8886 0.9189 0.9002 24256
5,0.243051,0.303003,precision recall f1-score support B-CONFLICT 0.0000 0.0000 0.0000 4 B-NEGATIVE 0.5000 0.2893 0.3665 121 B-NEUTRAL 0.0000 0.0000 0.0000 93 B-POSITIVE 0.4923 0.1429 0.2215 224 E-CONFLICT 0.0000 0.0000 0.0000 4 E-NEGATIVE 0.5000 0.2804 0.3593 107 E-NEUTRAL 0.0000 0.0000 0.0000 83 E-POSITIVE 0.5341 0.2304 0.3219 204 I-NEGATIVE 0.0000 0.0000 0.0000 30 I-NEUTRAL 0.0000 0.0000 0.0000 50 I-POSITIVE 0.0000 0.0000 0.0000 104 O 0.9370 0.9884 0.9620 22158 S-CONFLICT 0.0000 0.0000 0.0000 21 S-NEGATIVE 0.4949 0.3300 0.3960 297 S-NEUTRAL 0.2000 0.0049 0.0096 203 S-POSITIVE 0.4912 0.3526 0.4105 553 accuracy 0.9209 24256 macro avg 0.2593 0.1637 0.1905 24256 weighted avg 0.8886 0.9209 0.9012 24256


c:\Users\lalis\miniconda3\envs\ai_conda\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\lalis\miniconda3\envs\ai_conda\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\lalis\miniconda3\envs\ai_conda\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", 

Training completed and model saved successfully!


In [16]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from safetensors.torch import load_file

# =========================================================
# 1. DEVICE
# =========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================================================
# 2. LABELS (MUST MATCH TRAINING)
# =========================================================

labels = [
    "O",
    "B-POSITIVE", "I-POSITIVE", "E-POSITIVE", "S-POSITIVE",
    "B-NEGATIVE", "I-NEGATIVE", "E-NEGATIVE", "S-NEGATIVE",
    "B-NEUTRAL", "I-NEUTRAL", "E-NEUTRAL", "S-NEUTRAL",
    "B-CONFLICT", "I-CONFLICT", "E-CONFLICT", "S-CONFLICT"
]

label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}

# =========================================================
# 3. MODEL
# =========================================================

class BERT_BiLSTM(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()

        self.bert = AutoModel.from_pretrained(model_name)

        self.lstm = nn.LSTM(
            input_size=self.bert.config.hidden_size,
            hidden_size=256,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(512, num_labels)

    def forward(self, input_ids=None, attention_mask=None, **kwargs):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        x = outputs.last_hidden_state
        x, _ = self.lstm(x)
        x = self.dropout(x)
        logits = self.classifier(x)

        return logits

# =========================================================
# 4. LOAD MODEL + TOKENIZER
# =========================================================

model_path = "./absa_bilstm_model"
base_model = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = BERT_BiLSTM(base_model, len(labels))

state_dict = load_file(f"{model_path}/model.safetensors")
model.load_state_dict(state_dict)

model.to(device)
model.eval()

print("✅ Model loaded successfully")

# =========================================================
# 5. PREDICTION
# =========================================================

def predict(sentence):
    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    # remove token_type_ids (IMPORTANT FIX)
    inputs = {k: v.to(device) for k, v in inputs.items() if k != "token_type_ids"}

    with torch.no_grad():
        logits = model(**inputs)

    preds = torch.argmax(logits, dim=-1)[0].cpu().tolist()

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    tags = [id2label[p] for p in preds]

    return tokens, tags

# =========================================================
# 6. CLEAN TOKENS
# =========================================================

def clean_tokens(tokens):
    cleaned = []
    for t in tokens:
        if t in ["[CLS]", "[SEP]", "[PAD]"]:
            continue
        cleaned.append(t.replace("##", ""))
    return cleaned

# =========================================================
# 7. BIOES EXTRACTION
# =========================================================

def extract_aspects(tokens, tags):
    results = []
    i = 0

    while i < len(tags):
        tag = tags[i]

        # single token
        if tag.startswith("S-"):
            results.append((tokens[i], tag[2:]))
            i += 1

        # multi token span
        elif tag.startswith("B-"):
            label = tag[2:]
            start = i

            while i < len(tags) and not tags[i].startswith("E-"):
                i += 1

            if i < len(tags):
                phrase = " ".join(tokens[start:i+1])
                phrase = phrase.replace(" ##", "")
                results.append((phrase, label))

            i += 1

        else:
            i += 1

    return results

# =========================================================
# 8. RUN EXAMPLE
# =========================================================

sentence = "The food was amazing but the service was terrible"

tokens, tags = predict(sentence)
tokens = clean_tokens(tokens)

aspects = extract_aspects(tokens, tags)

print("\nTOKENS:")
print(tokens)

print("\nTAGS:")
print(tags)

print("\nASPECT SENTIMENTS:")
for a in aspects:
    print(a)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded successfully

TOKENS:
['the', 'food', 'was', 'amazing', 'but', 'the', 'service', 'was', 'terrible']

TAGS:
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

ASPECT SENTIMENTS:
